In [3]:
import pvdeg
import pvlib 
import xarray as xr
import pandas as pd
import numpy as np
import pytz
from pvdeg.decorators import geospatial_quick_shape
import matplotlib.pyplot as plt
import zarr

from dask.distributed import LocalCluster, Client
from dask_jobqueue import SLURMCluster

In [2]:
import sys, platform
print("Working on a ", platform.system(), platform.release())
print("Python version ", sys.version)
print("Pandas version ", pd.__version__)
print("Pvlib version: ", pvlib.__version__)
print("PVDeg version: ", pvdeg.__version__)

Working on a  Linux 4.18.0-477.10.1.el8_8.x86_64
Python version  3.9.21 | packaged by conda-forge | (main, Dec  5 2024, 13:51:40) 
[GCC 13.3.0]
Pandas version  2.2.3
Pvlib version:  0.10.3
PVDeg version:  0.3.5.dev245+gc25b2db.d20250205


In [ ]:
PVmodule = {
    "bifaciality": 0.70,
    "thermal_a": -3.47,
    "thermal_b": -0.0594,
    "thermal_dT": 3,
    "pdc0": 100,
    "gamma_pdc": -0.002,
}
trackingSystem = {
    "gcr": 0.4,
    "axis_tilt": 0,
    "axis_azimuth": 180,
    "backtracking": True,
    "max_angle": 55,
    "cross_axis_tilt": 0,
    "module_height": 1.5,
    "pitch": 5.7,
    "albedo": 0.2,
}

##### FUNCTION BODY ####
# lat = meta["latitude"]
# lon = meta["longitude"]
# tz = meta["tz"]

# weather_df = weather_df.tz_localize("UTC")

# try:
#     weather_df = weather_df.tz_convert(int(meta['tz'] * 3600))
#     #weather_df = weather_df.tz_convert('America/Denver')
# except TypeError: 
#     raise ValueError("localization failed")

# weather_df.head(24)

def offset_to_gmt(offset):
    sign = "+" if offset <= 0 else "-"
    return f"Etc/GMT{sign}{abs(offset)}"


def NSRDB_localize(weather_df, meta):
    zone_str = offset_to_gmt(meta['tz'])
    local_weather = weather_df.tz_localize("gmt").tz_convert(zone_str)
    return local_weather 

In [ ]:
getter = pvdeg.scenario.GeospatialScenario()
getter.addLocation(state="New York")
 
geo_weather, geo_meta = getter.weather_data, getter.meta_data

weather_df = geo_weather.isel(gid=0).to_dataframe()
meta = geo_meta.iloc[0].to_dict()

In [5]:
weather_db = "NSRDB"
weather_arg = {
    "satellite": "CONUS",
    "names": 2022,
    "NREL_HPC": True,
    "attributes": [
            "air_temperature",
            "wind_speed",
            "dhi",
            "ghi",
            "dni",
            "relative_humidity",
        ],
}

geo_weather, geo_meta = pvdeg.weather.get(
    weather_db, geospatial=True, **weather_arg
)

geo_weather


# geo_tmy_meta = geo_tmy_meta[geo_tmy_meta["state"] == "Colorado"]
# sub_tmy_meta, _ = pvdeg.utilities.gid_downsampling(geo_tmy_meta, 5)

# meta = sub_tmy_meta.iloc[20]
# weather_tmy_df = geo_tmy_weather.sel(gid=meta.name).to_dataframe()

/datasets/NSRDB/conus/nsrdb_conus_pv_2022.h5
/datasets/NSRDB/conus/nsrdb_conus_clouds_2022.h5


/home/tford/.conda-envs/geospatial/lib/python3.9/site-packages/xarray/core/dataset.py:277: UserWarning: The specified chunks separate the stored chunks along dimension "phony_dim_1" starting at index 500. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(


/datasets/NSRDB/conus/nsrdb_conus_csp_2022.h5
/datasets/NSRDB/conus/nsrdb_conus_ancillary_b_2022.h5
/datasets/NSRDB/conus/nsrdb_conus_irradiance_2022.h5
/datasets/NSRDB/conus/nsrdb_conus_clearsky_2022.h5
/datasets/NSRDB/conus/nsrdb_conus_ancillary_a_2022.h5


<xarray.Dataset> Size: 14TB
Dimensions:            (time: 105120, gid: 2842719)
Coordinates:
  * gid                (gid) int64 23MB 0 1 2 3 ... 2842716 2842717 2842718
  * time               (time) datetime64[ns] 841kB 2022-01-01 ... 2022-12-31T...
Data variables:
    temp_air           (time, gid) float64 2TB dask.array<chunksize=(105120, 500), meta=np.ndarray>
    wind_speed         (time, gid) float64 2TB dask.array<chunksize=(105120, 500), meta=np.ndarray>
    dhi                (time, gid) float64 2TB dask.array<chunksize=(105120, 500), meta=np.ndarray>
    ghi                (time, gid) float64 2TB dask.array<chunksize=(105120, 500), meta=np.ndarray>
    dni                (time, gid) float64 2TB dask.array<chunksize=(105120, 500), meta=np.ndarray>
    relative_humidity  (time, gid) float64 2TB dask.array<chunksize=(105120, 500), meta=np.ndarray>
Attributes:
    version:  4.0.0

In [6]:
geo_meta_us = geo_meta[geo_meta["country"]=="United States"]

geo_meta_us = geo_meta_us[geo_meta_us["state"]!="Hawaii"]
geo_meta_conus = geo_meta_us[geo_meta_us["state"]!="Alaska"]


geo_meta_conus

,latitude,longitude,altitude,tz,country,state,county,gid_full,wind_height
12864,48.380001,-124.730003,69,-8.0,United States,Washington,Clallam,444176,2
12865,48.160000,-124.730003,66,-8.0,United States,Washington,Clallam,444187,2
12866,48.380001,-124.709999,165,-8.0,United States,Washington,Clallam,444816,2
12867,48.180000,-124.709999,44,-8.0,United States,Washington,Clallam,444826,2
12868,48.160000,-124.709999,63,-8.0,United States,Washington,Clallam,444827,2
...,...,...,...,...,...,...,...,...,...
2825722,44.900002,-66.989998,24,-5.0,United States,Maine,Washington,6408612,2
2825723,44.880001,-66.989998,11,-5.0,United States,Maine,Washington,6408613,2
2825724,44.860001,-66.989998,12,-5.0,United States,Maine,Washington,6408614,2
2825725,44.820000,-66.989998,16,-5.0,United States,Maine,Washington,6408616,2


In [15]:
geo_meta_sub, gids = pvdeg.utilities.gid_downsampling(geo_meta_conus, 13)

geo_weather_sub = geo_weather.sel(gid=gids)

In [16]:
for var in geo_weather_sub.data_vars:
    

    geo_weather_sub[var].attrs.clear()

geo_weather_sub

<xarray.Dataset> Size: 16GB
Dimensions:            (time: 105120, gid: 3083)
Coordinates:
  * gid                (gid) int64 25kB 14563 14589 14617 ... 2824015 2824041
  * time               (time) datetime64[ns] 841kB 2022-01-01 ... 2022-12-31T...
Data variables:
    temp_air           (time, gid) float64 3GB dask.array<chunksize=(105120, 499), meta=np.ndarray>
    wind_speed         (time, gid) float64 3GB dask.array<chunksize=(105120, 499), meta=np.ndarray>
    dhi                (time, gid) float64 3GB dask.array<chunksize=(105120, 499), meta=np.ndarray>
    ghi                (time, gid) float64 3GB dask.array<chunksize=(105120, 499), meta=np.ndarray>
    dni                (time, gid) float64 3GB dask.array<chunksize=(105120, 499), meta=np.ndarray>
    relative_humidity  (time, gid) float64 3GB dask.array<chunksize=(105120, 499), meta=np.ndarray>
Attributes:
    version:  4.0.0

In [ ]:
geo_weather_sub.to_netcdf("/projects/pvsoiling/pvdeg/data/GOES_CONUS_5min/CONUS_5min.h5", engine="h5netcdf")



In [27]:
geo_meta_sub.to_csv("/projects/pvsoiling/pvdeg/data/GOES_CONUS_5min/meta.csv")

In [4]:
cluster = SLURMCluster(
    queue='shared',
    account="pvsoiling",
    cores=8,
    memory="80 GB",
    log_directory='/scratch/tford/dev/logs',
    walltime="02:00:00",
)
cluster.scale(12)
client = Client(cluster)
print(client.dashboard_link)

http://10.150.16.109:8787/status


In [ ]:
# Gracefully stop the Dask cluster
cluster.close()

# Disconnect the Dask client
client.close()

In [23]:
geo_weather_sub.chunk({"gid":100})

<xarray.Dataset> Size: 16GB
Dimensions:            (time: 105120, gid: 3083)
Coordinates:
  * gid                (gid) int64 25kB 14563 14589 14617 ... 2824015 2824041
  * time               (time) datetime64[ns] 841kB 2022-01-01 ... 2022-12-31T...
Data variables:
    temp_air           (time, gid) float64 3GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
    wind_speed         (time, gid) float64 3GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
    dhi                (time, gid) float64 3GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
    ghi                (time, gid) float64 3GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
    dni                (time, gid) float64 3GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
    relative_humidity  (time, gid) float64 3GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
Attributes:
    version:  4.0.0

In [25]:
import zarr

# Define Zarr-compatible encoding
encoding = {
    var: {
        "dtype": "float32",
        "compressor": zarr.Blosc(cname="zstd", clevel=4, shuffle=zarr.Blosc.SHUFFLE)
    }
    for var in geo_weather_sub.data_vars
}

# Write to Zarr with compression
geo_weather_sub.chunk({"gid":100}).to_zarr(
    "/projects/pvsoiling/pvdeg/data/GOES_CONUS_5min/CONUS_5min.zarr",
    encoding=encoding,
    compute=True
)


/home/tford/.conda-envs/geospatial/lib/python3.9/site-packages/distributed/client.py:3362: UserWarning: Sending large graph of size 48.02 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [26]:
xr.open_zarr("/projects/pvsoiling/pvdeg/data/GOES_CONUS_5min/CONUS_5min.zarr")

<xarray.Dataset> Size: 8GB
Dimensions:            (time: 105120, gid: 3083)
Coordinates:
  * gid                (gid) int64 25kB 14563 14589 14617 ... 2824015 2824041
  * time               (time) datetime64[ns] 841kB 2022-01-01 ... 2022-12-31T...
Data variables:
    dhi                (time, gid) float32 1GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
    dni                (time, gid) float32 1GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
    ghi                (time, gid) float32 1GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
    relative_humidity  (time, gid) float32 1GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
    temp_air           (time, gid) float32 1GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
    wind_speed         (time, gid) float32 1GB dask.array<chunksize=(105120, 100), meta=np.ndarray>
Attributes:
    version:  4.0.0

In [ ]:
encoding = {var: {"dtype": "float32", "_FillValue": -9999} for var in geo_weather_sub.data_vars}

write_task = geo_weather_sub.to_netcdf(
    "/projects/pvsoiling/pvdeg/data/GOES_CONUS_5min/CONUS_5min_encoded.h5",
    engine="h5netcdf",
    encoding=encoding,
    compute=False,
)

write_task.compute()

In [ ]:
weather_df = NSRDB_localize(weather_tmy_df, meta).head(24)

Updated Riccardio routine. Need to encapsulate into a function. Takeaway, we should use weather.get for this instead of geospatialScenario, the time-index of the scenario is wrong and provides us data which should be in UTC/GMT in something else (not sure what it is doing)

In [ ]:
time_index = weather_df.index

site_loc = pvlib.location.Location(meta["latitude"], meta["longitude"])
dni_clear = site_loc.get_clearsky(time_index)["dni"]
sp = site_loc.get_solarposition(time_index)

weather_df["Cdir"] = weather_df["dni"] / dni_clear

rotation = pvlib.tracking.singleaxis(
    apparent_zenith=sp["apparent_zenith"],
    apparent_azimuth=sp["azimuth"],
    axis_tilt=trackingSystem["axis_tilt"],
    axis_azimuth=trackingSystem["axis_azimuth"],
    backtrack=trackingSystem["backtracking"],
    gcr=trackingSystem["gcr"],
    max_angle=trackingSystem["max_angle"],
    cross_axis_tilt=trackingSystem["cross_axis_tilt"],
)

tilt_TT = rotation["surface_tilt"].fillna(0)

irrad_TT = pvlib.bifacial.infinite_sheds.get_irradiance(
    surface_tilt=tilt_TT,
    surface_azimuth=rotation["surface_azimuth"],
    solar_zenith=sp["apparent_zenith"],
    solar_azimuth=sp["azimuth"],
    gcr=trackingSystem["gcr"],
    height=trackingSystem["module_height"],
    pitch=trackingSystem["pitch"],
    ghi=weather_df["ghi"],
    dhi=weather_df["dhi"],
    dni=weather_df["dni"],
    albedo=trackingSystem["albedo"],
    bifaciality=PVmodule["bifaciality"],
)

temp_cell_TT = pvlib.temperature.sapm_cell(
    irrad_TT["poa_global"],
    temp_air=weather_df["temp_air"],
    wind_speed=weather_df["wind_speed"],
    a=PVmodule["thermal_a"],
    b=PVmodule["thermal_b"],
    deltaT=PVmodule["thermal_dT"],
)

pdc_TT = pvlib.pvsystem.pvwatts_dc(
    irrad_TT["poa_global"], temp_cell_TT, PVmodule["pdc0"], PVmodule["gamma_pdc"]
).fillna(0)

tilt_00 = pd.Series(0, index=tilt_TT.index)
irrad_00 = pvlib.bifacial.infinite_sheds.get_irradiance(
    surface_tilt=tilt_00,
    surface_azimuth=rotation["surface_azimuth"],
    solar_zenith=sp["apparent_zenith"],
    solar_azimuth=sp["azimuth"],
    gcr=trackingSystem["gcr"],
    height=trackingSystem["module_height"],
    pitch=trackingSystem["pitch"],
    ghi=weather_df["ghi"],
    dhi=weather_df["dhi"],
    dni=weather_df["dni"],
    albedo=trackingSystem["albedo"],
    bifaciality=PVmodule["bifaciality"],
)

temp_cell_00 = pvlib.temperature.sapm_cell(
    irrad_00["poa_global"],
    temp_air=weather_df["temp_air"],
    wind_speed=weather_df["wind_speed"],
    a=PVmodule["thermal_a"],
    b=PVmodule["thermal_b"],
    deltaT=PVmodule["thermal_dT"],
)
pdc_00 = pvlib.pvsystem.pvwatts_dc(
    irrad_00["poa_global"], temp_cell_00, PVmodule["pdc0"], PVmodule["gamma_pdc"]
).fillna(0)


GF_mask = weather_df["Cdir"] <= 0.1
pdc_GF = pdc_00.where(GF_mask, pdc_TT)
tilt_GF = tilt_00.where(GF_mask, tilt_TT)

#print('d', weather_df.dni.head(15))

mov_GF = tilt_GF.diff()
mov_TT = tilt_TT.diff()
mov_GF = mov_GF.dropna()
mov_TT = mov_TT.dropna()
mov_GF = np.abs(mov_GF)
mov_TT = np.abs(mov_TT)
mov_GF_cum = sum(mov_GF)
mov_TT_cum = sum(mov_TT)

energy_GF = sum(pdc_GF)  # /(60/30)
energy_TT = sum(pdc_TT)  # /(60/30)

# For debugging only -- remove afterwards
weather_df["dni_clear"] = dni_clear

weather_df['tilt_00'] = tilt_00
weather_df['pdc_00'] = pdc_00

weather_df['mov_TT'] = mov_TT
weather_df['pdc_TT'] = pdc_TT

weather_df['GF_mask'] = GF_mask
weather_df['tilt_GF'] = tilt_GF
weather_df['pdc_GF'] = pdc_GF
weather_df['mov_GF'] = mov_GF

weather_df.to_csv("internal_debug.csv")


# not sure if it is appropriate to set these to zeros instead of dividing by zero
try:
    delta_mov_perc = 100 * (mov_GF_cum - mov_TT_cum) / mov_TT_cum
except:
    delta_mov_perc = 0

try:
    delta_en_perc = 100 * (energy_GF - energy_TT) / energy_TT
except:
    delta_en_perc = 0

df_result = pd.DataFrame(
    {
        "delta_mov (%)": delta_mov_perc,
        "delta_en (%)": delta_en_perc,
        "energy_TT (Wh)": energy_TT,
    },
    index=[
        0,
    ],
)

df_result.head(24)

In [ ]:
# old riccardio method, use the cell above instead

time_index = weather_df.index

site_loc = pvlib.location.Location(lat, lon)
dni_clear = site_loc.get_clearsky(time_index)["dni"]
sp = site_loc.get_solarposition(time_index)

weather_df["Cdir"] = weather_df["dni"] / dni_clear


rotation = pvlib.tracking.singleaxis(
    apparent_zenith=sp["apparent_zenith"],
    apparent_azimuth=sp["azimuth"],
    axis_tilt=trackingSystem["axis_tilt"],
    axis_azimuth=trackingSystem["axis_azimuth"],
    backtrack=trackingSystem["backtracking"],
    gcr=trackingSystem["gcr"],
    max_angle=trackingSystem["max_angle"],
    cross_axis_tilt=trackingSystem["cross_axis_tilt"],
)

tilt_TT = rotation["surface_tilt"].fillna(0)

irrad_TT = pvlib.bifacial.infinite_sheds.get_irradiance(
    surface_tilt=tilt_TT,
    surface_azimuth=rotation["surface_azimuth"],
    solar_zenith=sp["apparent_zenith"],
    solar_azimuth=sp["azimuth"],
    gcr=trackingSystem["gcr"],
    height=trackingSystem["module_height"],
    pitch=trackingSystem["pitch"],
    ghi=weather_df["ghi"],
    dhi=weather_df["dhi"],
    dni=weather_df["dni"],
    albedo=trackingSystem["albedo"],
    bifaciality=PVmodule["bifaciality"],
)

temp_cell_TT = pvlib.temperature.sapm_cell(
    irrad_TT["poa_global"],
    temp_air=weather_df["temp_air"],
    wind_speed=weather_df["wind_speed"],
    a=PVmodule["thermal_a"],
    b=PVmodule["thermal_b"],
    deltaT=PVmodule["thermal_dT"],
)
pdc_TT = pvlib.pvsystem.pvwatts_dc(
    irrad_TT["poa_global"], temp_cell_TT, PVmodule["pdc0"], PVmodule["gamma_pdc"]
).fillna(0)


#print('b', weather_df.dni.head(15))


# ================= all year at 0deg (00) ==========================
tilt_00 = pd.Series(0, index=tilt_TT.index)
irrad_00 = pvlib.bifacial.infinite_sheds.get_irradiance(
    surface_tilt=tilt_00,
    surface_azimuth=rotation["surface_azimuth"],
    solar_zenith=sp["apparent_zenith"],
    solar_azimuth=sp["azimuth"],
    gcr=trackingSystem["gcr"],
    height=trackingSystem["module_height"],
    pitch=trackingSystem["pitch"],
    ghi=weather_df["ghi"],
    dhi=weather_df["dhi"],
    dni=weather_df["dni"],
    albedo=trackingSystem["albedo"],
    bifaciality=PVmodule["bifaciality"],
)
temp_cell_00 = pvlib.temperature.sapm_cell(
    irrad_00["poa_global"],
    temp_air=weather_df["temp_air"],
    wind_speed=weather_df["wind_speed"],
    a=PVmodule["thermal_a"],
    b=PVmodule["thermal_b"],
    deltaT=PVmodule["thermal_dT"],
)
pdc_00 = pvlib.pvsystem.pvwatts_dc(
    irrad_00["poa_global"], temp_cell_00, PVmodule["pdc0"], PVmodule["gamma_pdc"]
).fillna(0)

#print('c', weather_df.dni.head(15))



# ============== built the results of GoFlat routine (GF) ==========

GF_mask = weather_df["Cdir"] <= 0.1
pdc_GF = pdc_00.where(GF_mask, pdc_TT)
tilt_GF = tilt_00.where(GF_mask, tilt_TT)

#print('d', weather_df.dni.head(15))


mov_GF = tilt_GF.diff()
mov_TT = tilt_TT.diff()
mov_GF_cum = sum(mov_GF.dropna())
mov_TT_cum = sum(mov_TT.dropna())

#print('e', weather_df.dni.head(15))

energy_GF = sum(pdc_GF)  # /(60/30)
energy_TT = sum(pdc_TT)  # /(60/30)

weather_df['pdc_00'] = pdc_00
weather_df['tilt_00'] = tilt_00
weather_df['GF_mask'] = GF_mask
weather_df['pdc_GF'] = pdc_GF
weather_df['tilt_GF'] = tilt_GF
weather_df['mov_GF'] = mov_GF
weather_df['mov_TT'] = mov_TT

weather_df.to_csv("internal_debug.csv")

delta_mov_perc = 100 * (mov_GF_cum - mov_TT_cum) / mov_TT_cum
delta_en_perc = 100 * (energy_GF - energy_TT) / energy_TT


df_result = pd.DataFrame(
    {
        "delta_mov (%)": delta_mov_perc,
        "delta_en (%)": delta_en_perc,
        "energy_TT (Wh)": energy_TT,
    },
    index=[
        0,
    ],
)

In [ ]:
getter = pvdeg.scenario.GeospatialScenario()

#getter.addLocation(downsample_factor = 100)
#getter.downselect_CONUS()

getter.addLocation(state="Colorado")
 
geo_weather, geo_meta = getter.weather_data, getter.meta_data

In [ ]:
simulate_single(
    weather_df=geo_weather.isel(gid=0).to_dataframe(), 
    meta=geo_meta.iloc[0].to_dict(), 
    freq='1h', 
    PVmodule=PVmodule, 
    trackingSystem=trackingSystem
)

In [ ]:
# use this if it is less tha 10k points
workers = 8
 
cluster = LocalCluster(
    n_workers=workers,
    processes=True, 
)
 
client = Client(cluster)
 
print(client.dashboard_link)

In [ ]:
cluster = SLURMCluster(
    queue='shared',
    account="pvsoiling",
    cores=2,
    memory="30 GB",
    # processes=True,
    log_directory='/scratch/radinolf/dev/logs',
    walltime="02:00:00",
)
cluster.scale(32)
 
client = Client(cluster)
 
print(client.dashboard_link)

In [ ]:
geo_meta.iloc[0]

In [ ]:
template = pvdeg.geospatial.auto_template(func=simulate_single, ds_gids=geo_weather.isel(gid=slice(0,1)))

template

In [ ]:
template = pvdeg.geospatial.auto_template(func=simulate_single, ds_gids=geo_weather.isel(gid=slice(0,1)))r
res = pvdeg.geospatial.analysis(weather_ds = geo_weather.isel(gid=slice(0,1)), #geo_weather, 
                          meta_df = geo_meta.iloc[slice(0,1)], #geo_meta, 
                          func = simulate_single,
                          template = template,
                          freq = '1h', 
                          PVmodule = PVmodule, 
                          trackingSystem = trackingSystem)



In [ ]:
res

In [ ]:
pvdeg.geospatial.plot_sparse_analysis(
    result=res['energy_TT']
)

In [ ]:
weather_db = "NSRDB"
weather_arg = {
    "satellite": "Americas",
    "names": 2022,
    "NREL_HPC": True,
    "attributes": [
            "air_temperature",
            "wind_speed",
            "dhi",
            "ghi",
            "dni",
            "relative_humidity",
        ],
}

geo_weather, geo_meta = pvdeg.weather.get(
    weather_db, geospatial=True, **weather_arg
)

geo_weather_hr = geo_weather.sel(time=geo_weather.time[::2])

In [ ]:
geo_meta = geo_meta[geo_meta["state"] == "Colorado"]
sub_meta, sub_gids = pvdeg.utilities.gid_downsampling(geo_meta, 5)

sub_meta

In [ ]:
meta = sub_meta.iloc[0]

weather_df = geo_weather_hr.sel(gid=meta.name)

In [ ]:
start = 200*24
end = start + 24 + 24 # 2 days

weather_df.to_dataframe().iloc[start:end].dni.plot()

leftmax = weather_df.to_dataframe().iloc[start:end-24].dni.idxmax()
rightmax = weather_df.to_dataframe().iloc[start:end].dni.idxmax()

plt.axvline(x=leftmax, linewidth=1, color='k')
plt.axvline(x=rightmax, linewidth=1, color='k')
plt.title("utc time (CO Point) 2022")

print(leftmax)
print(rightmax)

In [ ]:
start = 200*24
end = start + 24 + 24 # 2 days

localize_df = NSRDB_localize(weather_df.to_dataframe(), meta)

localize_df.iloc[start:end].dni.plot()

leftmax = localize_df.iloc[start:end-24].dni.idxmax()
rightmax = localize_df.iloc[start:end].dni.idxmax()

plt.axvline(x=leftmax, linewidth=1, color='k')
plt.axvline(x=rightmax, linewidth=1, color='k')
plt.title("localized timezone (CO Point) 2022")

print(leftmax)
print(rightmax)